# Lecture 07 Policy

## 模块一：策略学习核心术语与符号定义

策略学习就是建立从“感知”到“动作”的映射。

*   **状态 (State, $s_t$)**: 环境的真实物理状态（通常在真实世界中不可见）。
*   **观测 (Observation, $o_t$)**: 外在传感器（如相机）和内在传感器（机械臂角度）获取到的信息。
*   **动作 (Action, $a_t$)**: 机器人执行的控制指令。
*   **策略 (Policy, $\pi$)**: 决策模型，通常用神经网络参数化为 $\theta$。
    *   **全观测 Fully observed 策略**: $\pi_\theta(a_t|s_t)$ 机器人一般做不到。
    *   **部分观测策略**: $\pi_\theta(a_t|o_t)$
*   **马尔可夫性质 (Markov Property)**: 下一步的状态只取决于当前状态和动作，与历史无关：$p(s_{t+1}|s_t, a_t)$ ，这其实就是 World Model，描述了世界的动力学 Dynamics。但真实情况中 $s_t$ 不一定考虑到了如精神状态等因素，所以不一定满足马尔可夫性。


---

## 模块二：模仿学习 (Imitation Learning, IL)

当存在最优策略或专家示教数据时，可以将策略学习转化为监督学习问题，即**行为克隆 (Behavioral Cloning, BC)**。

### 1. 核心思想
给定专家数据集 $\mathcal{D} = \{(o_i, a_i)\}$，通过监督学习直接拟合：$o_t \xrightarrow{\text{NN}} a_t$。
*   *数据来源*：人类遥操作 Teleoperation（如 Tesla、ALOHA 系统（一个人手拿着的 master arm，一个被遥控工作的 slave arm）、最优计算策略或教师策略。
*   当动作是离散的空间，本质和 Image classification 没有区别。是一种 supervised learning。
*   当动作是连续的空间，本质也是要 max log likelihood。

### 2. 致命缺陷：分布偏移 (Distributional Drift)
这是模仿学习的核心痛点（如 ALVINN 自动驾驶案例）。
*   **原因**：训练时，数据服从专家分布 $p_{\text{data}}(o_t)$；但在部署时，策略的微小误差会导致轨迹偏离，进入未见过的状态，即测试分布 $p_{\pi_\theta}(o_t) \neq p_{\text{data}}(o_t)$，导致误差累积（Error Compounding）并最终失败。

### 3. 解决方案 A：从数据层面解决 (收集更多数据)
**DAgger (Dataset Aggregation) 算法**
*   **目标**：强制让 $p_{\text{data}}(o_t) = p_{\pi_\theta}(o_t)$。
*   **流程**：
    1.  用人类数据 $\mathcal{D}$ 训练初始策略 $\pi_\theta$。
    2.  运行 $\pi_\theta$ 收集新的状态轨迹集 $\mathcal{D}_\pi$（让机器自己去跑，暴露出偏移的状态）。
    3.  **难点**：让人类专家对 $\mathcal{D}_\pi$ 中的每一个状态标注正确的动作 $a_t$。
    4.  合并数据集 $\mathcal{D} \leftarrow \mathcal{D} \cup \mathcal{D}_\pi$，重复步骤 1。
*   **问题**：让人类对机器犯错的中间状态进行补救标注是非常不自然的（Unnatural）。
*   现代 Dagger：让机器自己先做，做的快出问题了，人遥操接管，作为新增数据。这种时实接管，存在主臂与从臂在接管的一瞬间不一定处于同一状态的问题。可能的解决方案是利用Cartesian Space而非Configuration Space。
    | 维度 | Configuration Space（构型空间 / 关节空间） | Cartesian Space（笛卡尔空间 / 操作空间） |
    |:---|:---|:---|
    | **状态表示** | 关节角度/位置向量 `q = [θ₁, θ₂, ..., θₙ] ∈ ℝⁿ` | 末端执行器位姿 `X = [x, y, z, roll, pitch, yaw]` 或 `[p, q]` ∈ ℝ⁶ |
    | **控制对象** | 直接下发给电机/关节驱动器 | 作用于机器人末端在三维世界中的运动 |
    | **人类直觉** | 反直觉（“关节3转0.15rad”） | 符合直觉（“夹爪向左移5cm，俯仰转10°”） |
    | **映射关系** | 通过 **Forward Kinematics (FK)** → 末端位姿 | 通过 **Inverse Kinematics (IK)** → 关节指令 |
    | **典型场景** | 底层伺服、轨迹规划、动力学控制 | 遥操指令、任务级规划、人机交互 |

    **为什么 Cartesian + ΔR/ΔT + IK 能缓解/解决该问题？**  
    接管瞬间，系统**不依赖主控臂的绝对关节状态**，而是：
    - 实时读取从臂当前末端位姿 `X_current`（通过 FK 高频刷新）
    - 人类只需输入**相对增量** `ΔX = [ΔT, ΔR]`
    - 指令基准自动对齐到从臂真实状态，消除绝对跳变
    
    **为什么 Off-policy DAgger 存在“重复修正”（Repeated Correction）问题？**
    1. 状态访问分布偏移（Covariate Shift）
    - **On-policy DAgger**：每次训练的损失期望严格对齐当前策略的访问分布  
      $\mathcal{L}_{\text{on}} = \mathbb{E}_{s \sim d_{\pi_k}(s)}[\ell(\pi_\theta(s), a^*)]$
    - **Off-policy DAgger**：数据集 $D = \bigcup_{i=0}^{k-1} \{(s_t^i, a_t^{i*})\}$ 包含过去所有策略的观测。训练时通常直接做经验风险最小化：  
      $\mathcal{L}_{\text{off}} = \frac{1}{|D|}\sum_{(s,a^*)\in D} \ell(\pi_\theta(s), a^*)$
    - **问题**：$\pi_k$ 已经学会避开某些历史状态（如奇异点、易碰撞区域），但 $D$ 中仍大量存在这些状态。策略被迫**反复学习它已经不再访问的状态**，形成“刻舟求剑”式的重复修正。

    2. 梯度方向冲突与振荡
    - 不同历史策略 $\pi_i, \pi_j$ 在同一物理场景可能因轨迹不同而触发不同的专家干预。
    - 混合数据集的梯度是多个历史分布梯度的线性叠加：  
      $\nabla_\theta \mathcal{L}_{\text{off}} \approx \sum_i w_i \mathbb{E}_{s \sim d_{\pi_i}}[\nabla_\theta \ell]$
    - 当 $\pi_k$ 的实际轨迹分布 $d_{\pi_k}$ 与历史分布差异较大时，梯度会**在策略参数空间中来回拉扯**，表现为：修正状态A → 破坏状态B → 再次修正A，收敛缓慢甚至性能回退。


### 4. 解决方案 B：从模型层面解决 (提高策略拟合能力)
如果不增加数据，就需要模型极其完美地拟合专家，不产生初始偏差。传统 BC 失败的两个本质原因及对策：
*   **原因 1：非马尔可夫行为 (Non-Markovian Behavior)**
    *   *现象*：专家决策依赖历史信息，而模型只看当前帧。
    *   *对策*：引入历史帧。为了避免权重爆炸，使用共享权重的 **RNN / LSTM** 来提取历史状态。
    *   *隐患*：因果混淆 (Causal Confusion)——模型可能学到了虚假的因果关系（例如看到刹车灯亮才刹车，而不是看到行人刹车）。
*   **原因 2：多模态行为 (Multimodal Behavior)**
    *   *现象*：面对同一棵树，专家有时从左绕，有时从右绕。均方误差 (MSE) 会让模型取平均值（直接撞树）。
    *   *对策*：
        1. 输出高斯混合模型 (Mixture of Gaussians)。
        2. 离散化自回归模型 (Autoregressive discretization)：比如把走的方向分成左中右三个方向，然后变成一个classification问题，如果中间有障碍，那根据gt，模型最后学到的概率分布将会是0.5 0 0.5。
        3. 隐变量模型 (Latent variable models, 如 VAE)。
        4. **Diffusion Policy (扩散策略)**：通过去噪扩散过程生成动作，是目前具身智能（如机械臂操作）处理多模态分布的 SOTA 方法。

**L2 Loss 为什么隐含高斯/单峰假设？**  
L2 Loss 最小化均方误差，等价于在“条件高斯分布”假设下做极大似然估计；它天然输出条件期望 $\mathbb{E}[a|s]$，对多峰数据会做“模式平均（Mode Averaging）”，导致策略输出模糊、保守甚至危险的动作。

假设真实值 $y$ 与预测值 $\hat{y}$ 之间的误差 $\epsilon = y - \hat{y}$ 服从 **均值为 0、方差为 $\sigma^2$ 的高斯分布**：

$$
p(\epsilon) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{\epsilon^2}{2\sigma^2}\right)
$$

即给定输入 $x$ 时，输出 $y$ 的条件概率为：

$$
p(y \mid x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y - \hat{y})^2}{2\sigma^2}\right)
$$

对单个样本取**负对数似然**：

$$
-\log p(y \mid x) = \frac{1}{2\sigma^2} (y - \hat{y})^2 + \frac{1}{2}\log(2\pi\sigma^2)
$$

其中 $\frac{1}{2\sigma^2} > 0$ 为常数，$\frac{1}{2}\log(2\pi\sigma^2)$ 为与 $\hat{y}$ 无关的常数。

因此：
$$
\arg\min_{\hat{y}} \left[ -\log p(y \mid x) \right]
= \arg\min_{\hat{y}} \left[ \frac{1}{2\sigma^2} (y - \hat{y})^2 + \text{constant} \right]
= \arg\min_{\hat{y}} (y - \hat{y})^2
$$
即最小化负对数似然等价于最小化平方误差 $(y - \hat{y})^2$。

> 💡 关键点：L2 不关心数据有几个峰值，它只关心“平方误差最小”。当分布是多峰时，数学期望会落在**低概率区域**（两峰之间），策略被迫输出“折中动作”。


---

## 模块三：强化学习 (Reinforcement Learning, RL)

当没有专家示教，或者不知道什么是“正确”动作时，只能依靠**试错 (Trial and Error)**。
*   **与监督学习的区别**：数据不是独立同分布的（i.i.d.，当前动作影响未来状态）；没有 Ground Truth，只有成功/失败或延迟的奖励信号。

### 1. 数学形式化：马尔可夫决策过程 (MDP)
一个 MDP 定义为元组 $\mathcal{M} = \{\mathcal{S}, \mathcal{A}, \mathcal{T}, r\}$：
*   $\mathcal{S}$: 状态空间
*   $\mathcal{A}$: 动作空间
*   $\mathcal{T}$: 转移概率张量 $p(s_{t+1}|s_t, a_t)$
*   $r$: 奖励函数 $r(s_t, a_t) \to \mathbb{R}$
*   *(若是部分观测，则扩展为 POMDP，加入观测空间 $\mathcal{O}$ 和发射概率 $\mathcal{E}: p(o_t|s_t)$)*。

### 2. 强化学习的优化目标
定义一条交互轨迹 $\tau = (s_1, a_1, \dots, s_T, a_T)$，其发生的概率为：
$$p_\theta(\tau) = p(s_1) \prod_{t=1}^T \pi_\theta(a_t|s_t) p(s_{t+1}|s_t, a_t)$$

强化学习的目标是最大化**期望累计奖励**：
$$\theta^* = \arg\max_\theta J(\theta) = \arg\max_\theta \mathbb{E}_{\tau \sim p_\theta(\tau)} \left[ \sum_{t=1}^T r(s_t, a_t) \right]$$
*   *关键理解*：奖励函数 $r(x)$ 本身可能是不平滑的（如 0/1 阶跃信号），但**期望 $\mathbb{E}[r(x)]$ 关于参数 $\theta$ 是平滑的**，因此可以通过梯度下降优化。
